# VHAGAR T2 - Prithvi-EO-2.0-300M burn-scar fine-tune (Colab, tidy, no-restart)

**Run:** Runtime -> GPU, upload the data zip, then **Runtime -> Run all**. No kernel
restart needed (fit/test/predict run as subprocesses, so pip's numpy upgrade never
conflicts with the live kernel).

**Rebalanced run (recommended):** the first fine-tune under-detected burned area (IoU
~0.10, +0.05 skill vs the U-Net's +0.54) because the chips and pixels are mostly unburned.
Re-export the chips **burn-balanced** and train with a **Dice** loss:
```
vhagar t2-prithvi-export --cache-dir data\t2_prithvi --out-dir data\t2_prithvi_chips \
    --chip 224 --burn-balance
```
then set `LOSS = 'dice'` below. `ce` reproduces the original imbalanced run.


## 1. GPU + install (no restart)


In [ ]:
!nvidia-smi -L
!pip install -q terratorch "torchgeo==0.7.1" "numpy==2.2.6"
print('installed')


## 2. Parameters


In [ ]:
DATA_ZIP   = '/content/t2_prithvi_chips.zip'       # session upload, or a Drive path
MAX_EPOCHS = 60                                     # 3 for a smoke test
LOSS       = 'dice'                                 # 'dice' (imbalance-robust) | 'focal' | 'ce'
# from google.colab import drive; drive.mount('/content/drive')  # only if DATA_ZIP on Drive


## 3. Unzip the chips (fresh dir; shows the split sizes)


In [ ]:
!rm -rf /content/t2_prithvi_chips
!unzip -q -o "$DATA_ZIP" -d /content/t2_prithvi_chips
!echo images: $(ls /content/t2_prithvi_chips/data/*_merged.tif 2>/dev/null | wc -l) '| train:' $(wc -l < /content/t2_prithvi_chips/splits/train.txt 2>/dev/null)


## 4. Write the config (VHAGAR paths, data-derived normalization, LOSS from params)


In [ ]:
cfg = r'''seed_everything: 2
trainer:
  max_epochs: MAX_EPOCHS_PLACEHOLDER
  log_every_n_steps: 5
  callbacks:
    - class_path: EarlyStopping
      init_args: {monitor: val/loss, patience: 12}
    - class_path: ModelCheckpoint
      init_args: {monitor: val/loss, save_top_k: 1, filename: best}
  precision: bf16-mixed
model:
  class_path: terratorch.tasks.SemanticSegmentationTask
  init_args:
    model_factory: EncoderDecoderFactory
    model_args:
      backbone: prithvi_eo_v2_300
      backbone_pretrained: true
      backbone_bands: [BLUE, GREEN, RED, NIR_NARROW, SWIR_1, SWIR_2]
      necks:
        - {name: SelectIndices, indices: [5, 11, 17, 23]}
        - {name: ReshapeTokensToImage}
        - {name: LearnedInterpolateToPyramidal}
      decoder: UNetDecoder
      decoder_channels: [512, 256, 128, 64]
      num_classes: 2
    loss: LOSS_PLACEHOLDER
    ignore_index: -1
    freeze_backbone: false
    class_names: [Not burned, Burn scar]
optimizer:
  class_path: torch.optim.AdamW
  init_args: {lr: 1.e-4}
lr_scheduler:
  class_path: ReduceLROnPlateau
  init_args: {monitor: val/loss, factor: 0.5, patience: 4}
data:
  class_path: GenericNonGeoSegmentationDataModule
  init_args:
    batch_size: 8
    num_workers: 2
    dataset_bands: [BLUE, GREEN, RED, NIR_NARROW, SWIR_1, SWIR_2]
    output_bands: [BLUE, GREEN, RED, NIR_NARROW, SWIR_1, SWIR_2]
    rgb_indices: [2, 1, 0]
    train_data_root: /content/t2_prithvi_chips/data
    val_data_root: /content/t2_prithvi_chips/data
    test_data_root: /content/t2_prithvi_chips/data
    train_split: /content/t2_prithvi_chips/splits/train.txt
    val_split: /content/t2_prithvi_chips/splits/val.txt
    test_split: /content/t2_prithvi_chips/splits/test.txt
    img_grep: "*_merged.tif"
    label_grep: "*.mask.tif"
    means: [0.049771, 0.068685, 0.076481, 0.214572, 0.204851, 0.146201]
    stds:  [0.031063, 0.036317, 0.049595, 0.085017, 0.094316, 0.082628]
    num_classes: 2
    train_transform:
      - {class_path: albumentations.D4}
      - {class_path: ToTensorV2}
    test_transform:
      - {class_path: ToTensorV2}
    no_data_replace: 0
    no_label_replace: -1
'''.replace('MAX_EPOCHS_PLACEHOLDER', str(MAX_EPOCHS)).replace('LOSS_PLACEHOLDER', LOSS)
open('/content/prithvi_burnscars_vhagar.yaml','w').write(cfg)
print('config written: loss =', LOSS, '| max_epochs =', MAX_EPOCHS)


## 5. Fine-tune (GPU subprocess)


In [ ]:
!terratorch fit -c /content/prithvi_burnscars_vhagar.yaml


## 6. Native test metric (terratorch IoU/F1; watch test/IoU_Burn scar)


In [ ]:
import glob
ckpt = (sorted(glob.glob('/content/**/best*.ckpt', recursive=True)) or sorted(glob.glob('/content/**/*.ckpt', recursive=True)))[-1]
print('checkpoint:', ckpt)
!terratorch test -c /content/prithvi_burnscars_vhagar.yaml --ckpt_path "$ckpt"


## 7. Per-chip predictions on the test split (subprocess -> clean numpy)


In [ ]:
infer_src = r'''import glob, os, numpy as np, rasterio, torch
from terratorch.tasks import SemanticSegmentationTask
ck = sorted(glob.glob("/content/**/best*.ckpt", recursive=True)) or sorted(glob.glob("/content/**/*.ckpt", recursive=True))
assert ck, "no checkpoint - run fit first"
ckpt = ck[-1]; print("checkpoint:", ckpt)
task = SemanticSegmentationTask.load_from_checkpoint(ckpt, map_location="cuda").eval()
means = np.array([0.049771,0.068685,0.076481,0.214572,0.204851,0.146201], "float32")[:,None,None]
stds  = np.array([0.031063,0.036317,0.049595,0.085017,0.094316,0.082628], "float32")[:,None,None]
stems = [l.strip() for l in open("/content/t2_prithvi_chips/splits/test.txt") if l.strip()]
os.makedirs("/content/preds", exist_ok=True)
def to_logits(o):
    o = getattr(o, "output", o)
    return torch.as_tensor(o[0] if isinstance(o, (list, tuple)) else o)
for stem in stems:
    with rasterio.open(f"/content/t2_prithvi_chips/data/{stem}_merged.tif") as s:
        img = s.read().astype("float32")
    x = torch.from_numpy(((img - means) / stds)[None]).to("cuda")
    with torch.no_grad():
        pred = to_logits(task(x)).float().argmax(1)[0].cpu().numpy().astype("int16")
    with rasterio.open(f"/content/preds/{stem}.tif", "w", driver="GTiff",
                       height=pred.shape[0], width=pred.shape[1], count=1, dtype="int16") as d:
        d.write(pred[None])
print("wrote", len(stems), "per-chip prediction masks to /content/preds")
'''
open('/content/infer.py','w').write(infer_src)
!python /content/infer.py


## 8. Zip + download the predictions


In [ ]:
import shutil
shutil.make_archive('/content/prithvi_preds', 'zip', '/content/preds')
from google.colab import files; files.download('/content/prithvi_preds.zip')


## 9. Back on your machine: the head-to-head
```
Expand-Archive "$env:USERPROFILE\Downloads\prithvi_preds.zip" -DestinationPath data\prithvi_preds -Force
vhagar t2-prithvi-score --cache-dir data\t2_prithvi --pred-dir data\prithvi_preds \
    --chips-manifest data\t2_prithvi_chips\_chips.json
```
Per-fire skill-over-naive on the same test fires, vs the U-Net's +0.54.
